In [1]:
import numpy as np
import cv2
import os
import pywt
import joblib
from skimage.feature import hog, local_binary_pattern, graycomatrix, graycoprops
from skimage.color import rgb2gray

In [2]:
def fractal_dimension(Z, threshold=0.9):
    assert len(Z.shape) == 2
    Z = (Z < threshold)
    p = min(Z.shape)
    n = int(2**np.floor(np.log2(p)))
    sizes = 2**np.arange(int(np.log2(n)), 1, -1)
    counts = []
    for size in sizes:
        S = np.add.reduceat(
            np.add.reduceat(Z, np.arange(0, Z.shape[0], size), axis=0),
            np.arange(0, Z.shape[1], size), axis=1)
        counts.append(np.count_nonzero(S))
    coeffs = np.polyfit(np.log(sizes), np.log(counts), 1)
    return -coeffs[0]



In [3]:
def extract_texture_features(image, distances=[1], angles=[0, np.pi/2]):
    if image.dtype != np.uint8:
        image = (image * 255).astype(np.uint8)
    glcm = graycomatrix(image, distances, angles, 256, symmetric=True, normed=True)
    properties = ['contrast','homogeneity','energy','correlation']
    texture_features = []
    for prop in properties:
        vals = graycoprops(glcm, prop).flatten()
        texture_features.extend([np.mean(vals), np.std(vals)])
    return np.array(texture_features)


Wavelet features

In [4]:
def extract_wavelet_features(image, wavelet='db1', level=2):
    coeffs = pywt.wavedec2(image, wavelet=wavelet, level=level)
    features = []
    for coeff in coeffs:
        if isinstance(coeff, tuple):
            for arr in coeff:
                features.append(np.mean(arr))
                features.append(np.std(arr))
        else:
            features.append(np.mean(coeff))
            features.append(np.std(coeff))
    return np.array(features)



Explanation:
This function extracts wavelet features by decomposing the image into frequency components. For each coefficient array, we compute the mean and standard deviation


Morphological Features

In [5]:
def extract_morphological_features(image_path, resize_dim=(128,128)):
    img = cv2.imread(image_path)
    if img is None:
        return np.zeros(10)
    img = cv2.resize(img, resize_dim)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return np.zeros(10)
    c = max(contours, key=cv2.contourArea)
    area = cv2.contourArea(c)
    perimeter = cv2.arcLength(c, True)
    x,y,w,h = cv2.boundingRect(c)
    aspect_ratio = float(w)/h if h!=0 else 0
    circularity = 4*np.pi*area/(perimeter**2) if perimeter!=0 else 0
    extent = area/(w*h) if (w*h)!=0 else 0
    c = c.squeeze()
    if c.ndim!=2 or c.shape[0]<6:
        fourier_features = np.zeros(5)
    else:
        complex_contour = c[:,0] + 1j*c[:,1]
        fft_coeffs = np.fft.fft(complex_contour)
        if np.abs(fft_coeffs[1])==0:
            fourier_features = np.abs(fft_coeffs[1:6])
        else:
            fourier_features = np.abs(fft_coeffs[1:6])/np.abs(fft_coeffs[1])
    morph_features = np.array([area, perimeter, aspect_ratio, circularity, extent])
    return np.hstack([morph_features, fourier_features])



Explanation:
This function extracts shape descriptors: area, perimeter, aspect ratio, circularity, extent, plus Fourier descriptors of the contour.


Combined Feature extraction

In [6]:
def extract_all_features(image_path, resize_dim=(128,128)):
    img = cv2.imread(image_path)
    if img is None:
        return None
    img = cv2.resize(img, resize_dim)
    gray = rgb2gray(img)
    gray_uint8 = (gray*255).astype(np.uint8)

    hog_feat, _ = hog(gray_uint8, orientations=8, pixels_per_cell=(16,16),
                      cells_per_block=(1,1), visualize=True, channel_axis=None)

    lbp = local_binary_pattern(gray_uint8, P=8, R=1, method='uniform')
    n_bins = int(lbp.max()+1)
    lbp_hist, _ = np.histogram(lbp, bins=n_bins, range=(0,n_bins), density=True)

    chans = cv2.split(img)
    color_features = []
    for chan in chans:
        hist = cv2.calcHist([chan],[0],None,[32],[0,256])
        hist = cv2.normalize(hist,hist).flatten()
        color_features.extend(hist)
    color_features = np.array(color_features)

    morph_feat = extract_morphological_features(image_path, resize_dim)
    wavelet_feat = extract_wavelet_features(gray_uint8, wavelet='db1', level=2)
    texture_feat = extract_texture_features(gray_uint8)
    fract_dim = fractal_dimension(gray_uint8.astype(np.float32)/255.0, threshold=0.9)

    return np.hstack([hog_feat, lbp_hist, color_features, morph_feat, wavelet_feat, texture_feat, fract_dim])

Explanation:
This combines multiple feature types into one vector:
- HOG (edges/gradients)
- LBP (local texture)
- Color histograms
- Morphological + Fourier descriptors
- Wavelet features


In [7]:
base_path = r"C:\Users\Yaswanth\Downloads\diatom-classification\UDE Diatoms in the Wild 2024-Subset_n100\UDE Diatoms in the Wild 2024-Subset_n100\Top10_classes"

In [8]:
def extract_split_features(base_path, split):
    features, labels = [], []
    for species in os.listdir(base_path):
        species_path = os.path.join(base_path, species, split)
        if not os.path.isdir(species_path):
            continue
        for img_file in os.listdir(species_path):
            if img_file.lower().endswith(('.png','.jpg','.jpeg')):
                img_path = os.path.join(species_path, img_file)
                feat = extract_all_features(img_path)
                if feat is not None:
                    features.append(feat)
                    labels.append(species)
    return np.array(features), np.array(labels)

In [9]:
train_features, train_labels = extract_split_features(base_path, "train")
val_features, val_labels     = extract_split_features(base_path, "val")
test_features, test_labels   = extract_split_features(base_path, "test")

joblib.dump((train_features, train_labels), "train_features.pkl")
joblib.dump((val_features, val_labels), "val_features.pkl")
joblib.dump((test_features, test_labels), "test_features.pkl")

print("✅ Features saved as .pkl files")
print("Train:", train_features.shape, "Val:", val_features.shape, "Test:", test_features.shape)

✅ Features saved as .pkl files
Train: (14729, 651) Val: (4906, 651) Test: (4948, 651)
